In [58]:
from google.colab import files
uploaded = files.upload()  # CSV 파일 업로드 대화창이 뜹니다

Saving realworld_like_raw.csv to realworld_like_raw (1).csv


In [53]:
# # A_expand_sensitivity.py
# import pandas as pd
# import numpy as np

# # ---------- 입출력 ----------
# SRC = "realworld_like_raw.csv"                 # 민감도 없는 원본
# OUT_EXPANDED = "score_expanded_operational.csv"

# # ---------- 민감도 효과(정탐/오탐) 규칙 ----------
# k_lin_tdr   = 0.55   # 정탐률: 1~8 선형 상승 비중
# k_boost_tdr = 0.25   # 정탐률: 9에서 추가 상승
# a_lin_fa    = 0.08   # 오탐:   1~8 선형 증가율(단계당)
# b_boost_fa  = 0.35   # 오탐:   8→9 급증

# def tdr_adjust(p0, s):
#     p0 = float(np.clip(p0, 0.0, 0.99))
#     if s <= 8:
#         w = k_lin_tdr * (s - 1) / 7.0
#     else:  # s == 9
#         w = k_lin_tdr + k_boost_tdr
#     return float(np.clip(p0 + (1.0 - p0) * w, 0.0, 0.995))

# def fa_adjust(fa0, s):
#     fa0 = max(0.0, float(fa0))
#     inc = a_lin_fa * (s - 1) + b_boost_fa * max(0, s - 8)
#     return float(max(0.0, fa0 * (1.0 + inc)))

# # ---------- 로드 & 확장 ----------
# raw = pd.read_csv(SRC).reset_index(drop=False).rename(columns={"index":"RowID"})

# rows = []
# for _, r in raw.iterrows():
#     p0  = r["TrueDetectionRate"]
#     fa0 = r["FalseAlarms_25s"]
#     for s in range(1, 10):  # 민감도 1~9
#         rows.append({
#             "RowID": r["RowID"],
#             "ZoneID": r["ZoneID"],
#             "WindSpeed_kn": r["WindSpeed_kn"],
#             "WindDirection": r["WindDirection"],
#             "Season": r["Season"],
#             "Weather": r["Weather"],
#             "Distance_m": r["Distance_m"],
#             "Sensitivity": s,
#             "TrueDetectionRate_s": round(tdr_adjust(p0, s), 3),
#             "FalseAlarms_25s_s": round(fa_adjust(fa0, s), 3),
#         })

# expanded = pd.DataFrame(rows)
# expanded.to_csv(OUT_EXPANDED, index=False)
# print("✅ expanded saved:", OUT_EXPANDED, "rows:", len(expanded))


✅ expanded saved: score_expanded_operational.csv rows: 36000


In [54]:
# B_score_labeling_simple.py
import pandas as pd
import numpy as np

# 입력(전수표): raw×민감도(1~9) 확장 결과
IN_EXPANDED = "score_expanded_operational.csv"   # 필요시 파일명 변경
OUT_LABELS  = "score_labels_simple.csv"

# 1) 로드
expanded = pd.read_csv(IN_EXPANDED)

# 2) Score = 3*TD - 1*FA_Norm  (RowID 내 정규화)
EPS = 1e-9
expanded["FA_min"] = expanded.groupby("RowID")["FalseAlarms_25s_s"].transform("min")
expanded["FA_max"] = expanded.groupby("RowID")["FalseAlarms_25s_s"].transform("max")
expanded["FA_Norm"] = ((expanded["FalseAlarms_25s_s"] - expanded["FA_min"]) /
                       (expanded["FA_max"] - expanded["FA_min"] + EPS)).clip(0, 1)

expanded["Score"] = 3.0 * expanded["TrueDetectionRate_s"] - 1.0 * expanded["FA_Norm"]

# 3) 환경(RowID)별 최적 민감도 선택
def pick_best(g):
    return g.sort_values(
        ["Score", "FA_Norm", "TrueDetectionRate_s", "Sensitivity"],
        ascending=[False, True, False, True]
    ).iloc[0]

best = (expanded.groupby("RowID", as_index=False)
                  .apply(pick_best)
                  .reset_index(drop=True))

# 4) 결과 정리/저장
labels = best[[
    "RowID","ZoneID","WindSpeed_kn","WindDirection","Season","Weather","Distance_m",
    "Sensitivity","TrueDetectionRate_s","FalseAlarms_25s_s","FA_Norm","Score"
]].rename(columns={
    "Sensitivity":"OptimalSensitivity",
    "TrueDetectionRate_s":"TDR_at_Optimal",
    "FalseAlarms_25s_s":"FA_at_Optimal"
})

labels.to_csv(OUT_LABELS, index=False)

print("✅ saved:", OUT_LABELS, "rows:", len(labels))
print("distribution:", labels["OptimalSensitivity"].value_counts().sort_index().to_dict())
print(labels.head(10).to_string(index=False))


✅ saved: score_labels_simple.csv rows: 4000
distribution: {1: 1786, 7: 179, 8: 904, 9: 1131}
 RowID ZoneID  WindSpeed_kn WindDirection Season Weather  Distance_m  OptimalSensitivity  TDR_at_Optimal  FA_at_Optimal  FA_Norm    Score
     0    Z07         19.89            SW summer   clear        23.7                   9           0.867          19.90 1.000000 1.601000
     1    Z02         26.99             E summer   clear        26.2                   1           0.750          11.00 0.000000 2.250000
     2    Z02         13.57            SW spring   clear        22.9                   8           0.775           7.80 0.565657 1.759343
     3    Z01         18.78             W autumn  cloudy        25.9                   1           0.667           4.00 0.000000 2.001000
     4    Z08          4.07             W spring   clear        21.4                   1           0.667           1.00 0.000000 2.001000
     5    Z07          9.83            SE spring   clear        16.2           

/tmp/ipython-input-2359803477.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(pick_best)


In [55]:
# train_rf_optimal_sensitivity.py
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, top_k_accuracy_score
from sklearn.ensemble import RandomForestClassifier
import joblib

# ===== 경로 =====
LABELS_CSV = "score_labels_simple.csv"   # <- score_expanded_with_sensitivity에서 3:1로 라벨링한 결과
MODEL_OUT  = "rf_pipeline_optimal_sensitivity.joblib"

# ===== 데이터 로드 =====
df = pd.read_csv(LABELS_CSV)

# ===== 피처/타깃 정의 =====
FEATURES_CAT = ["ZoneID", "WindDirection", "Season", "Weather"]
FEATURES_NUM = ["WindSpeed_kn", "Distance_m"]
TARGET = "OptimalSensitivity"

# 누락 제거 및 타입 보정
df = df.dropna(subset=FEATURES_CAT + FEATURES_NUM + [TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)

X = df[FEATURES_CAT + FEATURES_NUM]
y = df[TARGET]

# ===== 전처리+모델 파이프라인 =====
ct = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), FEATURES_CAT),
        ("num", "passthrough", FEATURES_NUM),
    ]
)
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=4,
    random_state=42,
    n_jobs=-1
)
pipe = Pipeline([("prep", ct), ("rf", rf)])

# ===== 학습/검증 분리 & 학습 =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
pipe.fit(X_train, y_train)

# ===== 평가 =====
y_pred = pipe.predict(X_test)
proba  = pipe.predict_proba(X_test)
classes_ = pipe.named_steps["rf"].classes_

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

# top-2 accuracy (두 후보 안에 정답이 포함될 확률)
print("Top-2 Accuracy:", round(top_k_accuracy_score(y_test, proba, k=2, labels=classes_), 4))

# 가벼운 교차검증(3-fold)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
print("CV(3-fold) Acc:", [round(s,4) for s in cv_scores], "-> mean:", round(cv_scores.mean(),4))

# ===== 저장 =====
joblib.dump(pipe, MODEL_OUT)
print("Saved model to:", MODEL_OUT)


Accuracy: 0.4738
[[264   2  40  51]
 [ 10   1   1  24]
 [126   0  29  26]
 [112   1  28  85]]
              precision    recall  f1-score   support

           1       0.52      0.74      0.61       357
           7       0.25      0.03      0.05        36
           8       0.30      0.16      0.21       181
           9       0.46      0.38      0.41       226

    accuracy                           0.47       800
   macro avg       0.38      0.33      0.32       800
weighted avg       0.44      0.47      0.44       800

Top-2 Accuracy: 0.75
CV(3-fold) Acc: [np.float64(0.4738), np.float64(0.4666), np.float64(0.4771)] -> mean: 0.4725
Saved model to: rf_pipeline_optimal_sensitivity.joblib


In [57]:
# demo_predict.py
import pandas as pd
import joblib

MODEL_PATH = "rf_pipeline_optimal_sensitivity.joblib"
pipe = joblib.load(MODEL_PATH)

# 예: 실시간 환경 입력 (3건)
demo = pd.DataFrame([
    {"ZoneID":"Z03","WindDirection":"NE","Season":"summer","Weather":"rain","WindSpeed_kn":18.5,"Distance_m":22.0},
    {"ZoneID":"Z07","WindDirection":"W","Season":"autumn","Weather":"clear","WindSpeed_kn":9.0,"Distance_m":14.5},
    {"ZoneID":"Z01","WindDirection":"S","Season":"winter","Weather":"snow","WindSpeed_kn":26.0,"Distance_m":35.0},
])

pred = pipe.predict(demo)
proba = pipe.predict_proba(demo)
classes_ = pipe.named_steps["rf"].classes_

# 상위 2개 후보 및 확률까지 보기
top2 = []
for row in proba:
    top2.append(sorted(zip(classes_, row), key=lambda t: t[1], reverse=True)[:2])

out = demo.copy()
out["Predicted_OptimalSensitivity"] = pred
out["Top2"] = top2
print(out)


  ZoneID WindDirection  Season Weather  WindSpeed_kn  Distance_m  \
0    Z03            NE  summer    rain          18.5        22.0   
1    Z07             W  autumn   clear           9.0        14.5   
2    Z01             S  winter    snow          26.0        35.0   

   Predicted_OptimalSensitivity  \
0                             1   
1                             1   
2                             1   

                                                Top2  
0  [(1, 0.49148095238095246), (8, 0.2613976190476...  
1  [(1, 0.7452857142857147), (8, 0.18818095238095...  
2  [(1, 0.5024968253968253), (8, 0.34459841269841...  
